## Shamir scret sharing with Chinese Remainder Theorem 
This is the Asmuth–Bloom approach described in https://en.wikipedia.org/wiki/Secret_sharing_using_the_Chinese_remainder_theorem

In [1]:
import numpy as np
import random as rd
from math import gcd

SETUP:

the setting is a 3-party scenario. This means that $n = 3$. This is because $n$ refers to the number of parties that receive share. For the threshold instead, we decide to set it to $k = 2$. The secret is 50. 

In [2]:
n = 3
k = 2
S = 50

1. we compute the moduli we will need to make the shares. The moduli need to be pairwise coprime and, on top of that, need to satisfy some important conditions:
\begin{align*}
S < \Pi_{i=1}^k m_i \\
 \\
m_0 < ... < m_n  \\
 \\
m_0 \cdot m_{n-k+2} \cdots m_n < m_1 \cdots m_k
\end{align*}

In [3]:
from math import gcd

def check_moduli(moduli, n, k):
    # Check increasing
    if not all(moduli[i] < moduli[i + 1]
               for i in range(len(moduli) - 1)):
        return False

    # Check pairwise coprime
    if not all(gcd(moduli[i], moduli[j]) == 1
               for i in range(len(moduli))
               for j in range(i + 1, len(moduli))):
        return False

    # Check Asmuth-Bloom inequality
    left = moduli[0]
    for i in range(n - k + 2, n + 1):
        left *= moduli[i]

    right = 1
    for i in range(1, k + 1):
        right *= moduli[i]

    return left < right

moduli = [53, 59, 61, 67]

if check_moduli(moduli, 3, 2):
    print("The moduli are valid")
else:
    print("The moduli are not valid")

The moduli are valid


The moduli $w_0 = 53, w_1 = 59,w_2 = 61, w_3 = 67$ are pairwise coprime positive integers and satisfy the conditions.In fact, 

\begin{align*}
50 < 59 \cdot 61  \\
\\
53 < 59 < 61 < 67  \\
 \\
53 \cdot 67 < 59 \cdot 61
\end{align*}

2. Find $\alpha$. It is a random number that satifies the following criterion:

\begin{equation}
    S + \alpha \cdot m_0 < m_1 \cdots m_k
\end{equation} 
We chose $\alpha = 4$. In fact $50 + 4 \cdot 53 < 59 \cdot 61 \cdot 67$

3. Compute the shares: 

$S + \alpha \cdot m_0 \mod m_i$ for $1 \leq i \leq n $ 

We define $S + \alpha \cdot m_0$ as $S_0$


The shares are returned as $I_i = (s_i, m_i)$ 

In [5]:
alpha = 4
shares = []

def find_share(alpha):
    for i in (moduli[1:]):
        shares.append((S + alpha * moduli[0]) % i)
find_share(alpha)

shares = list(zip(shares, moduli[1:]))
print(f"the shares are {shares}")

the shares are [(26, 59), (18, 61), (61, 67)]


## Reconstruction

4. We can choose any set of $k$ shares and set them into a system:


\begin{cases}
x \equiv & 26 \mod 59 \\
x \equiv & 18 \mod 61
\end{cases}

At this point, we have the minimum information to find the solution of the system: 

$S_0 \mod m_{i_1} \cdots m_{i_k} \rightarrow S_0 \mod 3599$ 

However, from the parties perspective, there are many possible solutions. They need a way to find the only solution to the system

We used the approach explained on Wikipedia, using Bezout's identity and the Extented Euclidian Algorithm

5. Since the parties do not know $S_0$, we want to reconstruct it:
We can find two numbers $e_1$ and $e_2$ such that:
\begin{align*}
e_1 &\equiv 1 \mod{59}, & e_1 &\equiv 0 \mod{61}, \
e_2 &\equiv 0 \mod{59}, & e_2 &\equiv 1 \mod{61}.
\end{align*}
In such a way, we can reconstruct $S_0$: 

$S_0 = s_1 \cdot e_1 + s_2 \cdot e_2$. 

Remember that $s_1 = 26$ and $s_2 = 18$. 

$e_1$ and $e_2$ guarantee that $S_0 \equiv 26 \mod59$ and $S_0 \equiv 18 \mod61$. 

6. find $e_1$

$e_1 = 61 \cdot y_1$ 

We want to find:

$61 \cdot y_1 \equiv 1 \mod59$, we know that $61 =  59 \cdot 1 +2$

thus, $61 \equiv 2 \mod59$

Plug this back into $61 \cdot y_1 \equiv 1 \mod59$

$2 \cdot y_1 \equiv 1 \mod 59$

thus, $y_1 = 30$

$e_1 = 61 \cdot y_1 = 1830$

7. find $e_2$

set $e_2 = 59 \cdot y_2$ so that $e_2$ is multiple of 59

find $59 \cdot y_2 \equiv 1 \mod 61$

we know $59 \equiv -2 \mod 61$

thus, $-2 \cdot y_2 \equiv 1 \mod 61$

so, $y_2 = 30$

$e_2 = 59 \cdot y_2 = 1770$

8. Now the parties can compute $S_0$

$S_0 = s_1 \cdot e_1 + s_2 \cdot e_2 = 26 \cdot 1830 + 18 \cdot 1770 = 79440$

Now, $79440 \mod (59 \cdot 61)$

The result is 262, which is exactly the $S_0$ that we found during the encryption

9. Finally, the parties can compute the secret $S$:

Considering that the parameter $m_0$ is the only one that is public and known by all the parties, they have now all the information to compute the secret $S$:

$S = S_0 \mod m_0$

$S = 262 \mod 53$ 

$50$

We successfully recovered the secret